In [69]:
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import r2_score
import time
from sklearn.datasets import make_sparse_spd_matrix
from scipy.stats import multivariate_normal
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import KFold
import pandas as pd
import pickle
import os
from datetime import datetime

In [82]:
# Default functions
def mu_0_default(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e_default(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau_default(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]


class DGP:
    def __init__(self, n=2000, p=20, alpha=0.85, mu_0=mu_0_default, tau=tau_default, e=e_default, sigma_epsilon=1, seed=0):
        """
        Data Generating Process (DGP) class initializer.

        Parameters:
        n (int): Number of samples
        p (int): Number of features
        alpha (float): Sparsity parameter for the covariance matrix
        mu_0 (function): Baseline outcome function
        tau (function): Treatment effect function
        e (function): Propensity score function
        sigma_epsilon (float): Standard deviation of the error terms
        seed (int): Random seed for reproducibility
        """
        self.n = n
        self.p = p
        self.alpha = alpha
        self.mu_0 = mu_0
        self.tau = tau
        self.e = e
        self.sigma_epsilon = sigma_epsilon
        self.seed = seed

    def generate_data(self):
        """
        Generate synthetic data.

        Returns:
        np.ndarray: Observed data matrix containing features, observed outcomes, and treatment assignments
        """
        np.random.seed(self.seed)

        # Generate the correlation matrix
        Sigma = make_sparse_spd_matrix(norm_diag=True, n_dim=self.p, alpha=self.alpha)

        # Generate covariates X
        X = multivariate_normal.rvs(mean=np.zeros(self.p), cov=Sigma, size=self.n)

        # Generate noise-free potential outcomes
        M0 = self.mu_0(X)
        M1 = M0 + self.tau(X)

        # Generate noise
        epsilon0 = np.random.normal(0, self.sigma_epsilon, self.n)
        epsilon1 = np.random.normal(0, self.sigma_epsilon, self.n)

        # Generate noisy potential outcomes
        Y0 = M0 + epsilon0
        Y1 = M1 + epsilon1

        # Generate treatment assignments
        T = np.random.binomial(1, self.e(X))

        # Generate observed outcomes
        Y = T * Y1 + (1 - T) * Y0

        # Combine into observed data matrix
        D_matrix = np.hstack((X, Y.reshape(-1, 1), T.reshape(-1, 1)))
        return D_matrix

    @staticmethod
    def calculate_density(matrix, threshold=1e-5):
        """Calculate the density of non-zero elements in a matrix."""
        non_zero_elements = np.abs(matrix) > threshold
        density = np.sum(non_zero_elements) / matrix.size
        return density

    def mean_density(self, alpha, iter=1000):
        """Calculate the mean density for a given alpha over several iterations."""
        densities = []
        for _ in range(iter):
            matrix = make_sparse_spd_matrix(norm_diag=True, n_dim=self.p, alpha=alpha)
            densities.append(self.calculate_density(matrix))
        return np.mean(densities)

    def find_alpha_for_target_density(self, target_density=0.3, tolerance=0.01, iter_per_alpha=1000):
        """Find the alpha value that produces a matrix with the target density."""
        best_alpha = None
        best_density_diff = float('inf')

        for alpha in np.arange(0, 1.01, 0.01):
            density = self.mean_density(alpha, iter_per_alpha)
            print(density)
            density_diff = abs(density - target_density)

            if density_diff < best_density_diff:
                best_density_diff = density_diff
                best_alpha = alpha

            if density_diff < tolerance:
                break
        return best_alpha, self.mean_density(best_alpha, iter_per_alpha)

    def set_alpha_from_density(self, target_density=0.3):
        self.alpha = self.find_alpha_for_target_density(target_density)[0]

    def tune_confounding(self, c1_e_values, c2_e_values, c1_mu_values, c2_mu_values, target_r2_ps, target_r2_outcome, tolerance=0.05):
        """
        Tune parameters and measure R^2 for both propensity score and outcome model.

        Parameters:
        X (np.ndarray): Covariates
        c1_e_values (list): Values for c1_e
        c2_e_values (list): Values for c2_e
        c1_mu_values (list): Values for c1_mu
        c2_mu_values (list): Values for c2_mu
        target_r2_ps (float): Target R^2 for propensity score model
        target_r2_outcome (float): Target R^2 for outcome model
        tolerance (float): Tolerance for R^2 matching

        Returns:
        list: Results containing the optimal parameters and their corresponding R^2 values
        """
        results = []
        for c1_e in c1_e_values:
            for c2_e in c2_e_values:
                for c1_mu in c1_mu_values:
                    for c2_mu in c2_mu_values:
                        self.e = lambda X: e_default(X, c1_e, c2_e)
                        self.mu_0 = lambda X: mu_0_default(X, c1_mu, c2_mu)
                        data = self.generate_data()
                        X = data[:,:-2]
                        Y = data[:,-2]
                        T = data[:, -1]
                        # Fit logistic regression model to predict treatment assignment
                        log_reg = LogisticRegression().fit(X, T)
                        T_pred = log_reg.predict_proba(X)[:, 1]
                        print(T-T_pred)
                        r2_ps = r2_score(T, T_pred)
                        lin_reg = LinearRegression().fit(X[T == 0], Y[T == 0])
                        Y_pred = lin_reg.predict(X[T == 0])
                        r2_outcome = r2_score(Y[T == 0], Y_pred)
                        # Check if the R^2 values are within the desired range
                        if (abs(r2_ps - target_r2_ps) <= tolerance) and (abs(r2_outcome - target_r2_outcome) <= tolerance):
                            results.append({'c1_e': c1_e, 'c2_e': c2_e, 'c1_mu': c1_mu, 'c2_mu': c2_mu, 'R^2_ps': r2_ps, 'R^2_outcome': r2_outcome})
        return results

    @staticmethod
    def find_optimal_parameters(results, target_r2_ps, target_r2_outcome):
        """
        Find the optimal parameters based on the closest R^2 values to the targets.

        Parameters:
        results (list): List of dictionaries containing parameters and their corresponding R^2 values
        target_r2_ps (float): Target R^2 for propensity score model
        target_r2_outcome (float): Target R^2 for outcome model

        Returns:
        dict: Optimal parameters with their corresponding R^2 values
        """
        optimal_result = None
        min_diff = float('inf')

        for result in results:
            diff = abs(result['R^2_ps'] - target_r2_ps) + abs(result['R^2_outcome'] - target_r2_outcome)
            if diff < min_diff:
                min_diff = diff
                optimal_result = result
        return optimal_result

    def find_parameters(self, level='almost_no', parameter_spaces = None):
        """
        Find optimal parameters for different levels of confounding.

        Parameters:
        parameter_spaces (dict, optional): Parameter spaces for different levels of confounding

        Returns:
        dict: Optimal parameters for each level of confounding
        """
        # Define different parameter spaces for each level of confounding
        if level == 'almost_no':
            params = {
                "c1_e_values": np.linspace(0.01, 0.5, 5),
                "c2_e_values": np.linspace(0.01, 0.5, 5),
                "c1_mu_values": np.linspace(1.5, 3, 5),
                "c2_mu_values": np.linspace(0.5, 1.5, 5),
                "target_r2_ps": 0.05,
                "target_r2_mu_0": 0.5
            }
        elif level == 'low':
            params = {
                "c1_e_values": np.linspace(0.1, 1.5, 5),
                "c2_e_values": np.linspace(0.1, 1.5, 5),
                "c1_mu_values": np.linspace(1.5, 3, 5),
                "c2_mu_values": np.linspace(0.5, 1.5, 5),
                "target_r2_ps": 0.2,
                "target_r2_mu_0": 0.5
            }
        elif level == 'mid':
            params = {
                "c1_e_values": np.linspace(1.5, 3, 5),
                "c2_e_values": np.linspace(0.3, 0.5, 5),
                "c1_mu_values": np.linspace(1.5, 3, 5),
                "c2_mu_values": np.linspace(0.5, 1.5, 5),
                "target_r2_ps": 0.5,
                "target_r2_mu_0": 0.5
            }
        elif level == 'high':
            params = {
                "c1_e_values": np.linspace(0.1, 10, 5),
                "c2_e_values": np.linspace(0.1, 6, 5),
                "c1_mu_values": np.linspace(1.5, 3, 5),
                "c2_mu_values": np.linspace(0.5, 1.5, 5),
                "target_r2_ps": 0.8,
                "target_r2_mu_0": 0.5
            }
        elif level == 'test':
            params = {
                "c1_e_values": np.linspace(0.3725, 0.3725, 1),
                "c2_e_values": np.linspace(-10., -10., 1),
                "c1_mu_values": np.linspace(1.5, 3, 5),
                "c2_mu_values": np.linspace(0.5, 1.5, 5),
                "target_r2_ps": 0.05,
                "target_r2_mu_0": 0.5
            }
        else:
            params = parameter_spaces

        results = self.tune_confounding(params["c1_e_values"], params["c2_e_values"], params["c1_mu_values"], params["c2_mu_values"], params["target_r2_ps"], params["target_r2_mu_0"])
        optimal_parameters = self.find_optimal_parameters(results, params["target_r2_ps"], params["target_r2_mu_0"])
        return optimal_parameters

    def set_optimal_parameters(self, confounding_level="almost_no"):
        """
        Set the parameters self.e and self.mu_0 using the optimal parameter search.

        Parameters:
        X (np.ndarray): Covariates
        confounding_level (str): Level of confounding ('almost_no', 'low', 'mid', 'high')
        """
        optimal = self.find_parameters(level=confounding_level)
        print(optimal)
        if optimal:
            self.e = lambda X: e_default(X, optimal['c1_e'], optimal['c2_e'])
            self.mu_0 = lambda X: mu_0_default(X, optimal['c1_mu'], optimal['c2_mu'])
        else:
            raise ValueError("Optimal parameters not found for the specified confounding level")

In [71]:
tree_settings_init = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

class S_learner:
    def __init__(self, tree_settings=tree_settings_init,K=1,median=True):
        self.tree_settings = tree_settings
        self.K = K
        self.median = median

    def get_tau(self,D):
        if self.K == 1:
            model1 = self.get_model()
            model1.fit(X=np.delete(D, -2, axis=1),y=D[:,-2])
            Y1_pred = model1.predict(np.hstack((D[:,:-2], np.ones((D.shape[0],1)))))
            Y0_pred = model1.predict(np.hstack((D[:,:-2], np.zeros((D.shape[0],1)))))
            Y_pseudo = Y1_pred-Y0_pred
            model2 = self.get_model()
            model2.fit(X=D[:,:-2],y=Y_pseudo)
            tau = lambda X: model2.predict(X=X)
            return tau
        else:
            kf = KFold(n_splits=self.K, shuffle=True)
            folds = [indices for _,indices in kf.split(D)]
            for k in range(self.K):
                taus = []
                D_test = D[folds[k]]
                D_train = D[folds[(k + 1) % self.K]]
                model1 = self.get_model()
                model1.fit(X=np.delete(D_train, -2, axis=1),y=D_train[:,-2])
                Y1_pred = model1.predict(np.hstack((D_test[:,:-2], np.ones((D_test.shape[0],1)))))
                Y0_pred = model1.predict(np.hstack((D_test[:,:-2], np.zeros((D_test.shape[0],1)))))
                Y_pseudo = Y1_pred-Y0_pred
                model2 = self.get_model()
                model2.fit(X=D_test[:,:-2],y=Y_pseudo)
                tau_k = lambda X: model2.predict(X=X)
                taus.append(tau_k)
            if self.median:
                tau = lambda X: np.median(np.array([tau_k(X) for tau_k in taus]), axis=0)
            else:
                print('np.mean')
                tau = lambda X: np.mean(np.array([tau_k(X) for tau_k in taus]), axis=0)
            return tau

    def get_model(self):
        model = RandomForestRegressor(
                n_estimators=self.tree_settings['n_trees'],
                max_features=self.tree_settings['max_features'],
                min_samples_leaf=self.tree_settings['min_samples_leaf'],
                max_depth=self.tree_settings['max_depth'],
                n_jobs=-1)
        return model

In [72]:
class T_learner:
    def __init__(self, tree_settings=tree_settings_init,K=1,median=True):
        self.median = median
        self.tree_settings = tree_settings
        self.K = K

    def get_tau(self,D):
        if self.K == 1:
            model1 = self.get_model()
            model1.fit(X=D[D[:,-1] == 1,:-2],y=D[D[:,-1] == 1,-2])
            Y1_pred = model1.predict(D[:,:-2])
            model2 = self.get_model()
            model2.fit(X=D[D[:,-1] == 0,:-2],y=D[D[:,-1] == 0,-2])
            Y0_pred = model2.predict(D[:,:-2])
            Y_pseudo = Y1_pred-Y0_pred
            model3 = self.get_model()
            model3.fit(X=D[:,:-2],y=Y_pseudo)
            tau = lambda X: model3.predict(X=X)
            return tau
        else:
            kf = KFold(n_splits=self.K, shuffle=True)
            folds = [indices for _,indices in kf.split(D)]
            for k in range(self.K):
                taus = []
                D_test = D[folds[k]]
                D_train1 = D[folds[(k + 1) % self.K]]
                D_train2 = D[folds[(k + 2) % self.K]]
                model1 = self.get_model()
                model1.fit(X=D_train1[D_train1[:,-1] == 1,:-2],y=D_train1[D_train1[:,-1] == 1,-2])
                Y1_pred = model1.predict(D_test[:,:-2])
                model2 = self.get_model()
                model2.fit(X=D_train2[D_train2[:,-1] == 0,:-2],y=D_train2[D_train2[:,-1] == 0,-2])
                Y0_pred = model2.predict(D_test[:,:-2])
                Y_pseudo = Y1_pred-Y0_pred
                model3 = self.get_model()
                model3.fit(X=D_test[:,:-2],y=Y_pseudo)
                tau_k = lambda X: model3.predict(X=X)
                taus.append(tau_k)
            if self.median:
                tau = lambda X: np.median(np.array([tau_k(X) for tau_k in taus]), axis=0)
            else:
                tau = lambda X: np.mean(np.array([tau_k(X) for tau_k in taus]), axis=0)
            return tau

    def get_model(self):
        model = RandomForestRegressor(
                n_estimators=self.tree_settings['n_trees'],
                max_features=self.tree_settings['max_features'],
                min_samples_leaf=self.tree_settings['min_samples_leaf'],
                max_depth=self.tree_settings['max_depth'],
                n_jobs=-1)
        return model

In [73]:
class IPW_learner:
    def __init__(self, tree_settings=tree_settings_init,K=1,median=True):
        self.median = median
        self.tree_settings = tree_settings
        self.K = K

    def get_tau(self,D):
        if self.K == 1:
            model1 = self.get_model(regressor=False)
            model1.fit(X=D[:,:-2],y=D[:,-1])
            e_pred = np.clip(model1.predict_proba(D[:,:-2])[:,1], 0.025, 0.975)
            Y_pseudo = D[:,-2]*(D[:,-1]/e_pred - (1-D[:,-1])/(1-e_pred))
            model2 = self.get_model()
            model2.fit(X=D[:,:-2],y=Y_pseudo)
            tau = lambda X: model2.predict(X=X)
            return tau
        else:
            kf = KFold(n_splits=self.K, shuffle=True)
            folds = [indices for _,indices in kf.split(D)]
            for k in range(self.K):
                taus = []
                D_test = D[folds[k]]
                D_train = D[folds[(k + 1) % self.K]]
                model1 = self.get_model(regressor=False)
                model1.fit(X=D_train[:,:-2],y=D_train[:,-1])
                e_pred = np.clip(model1.predict_proba(D_test[:,:-2])[:,1], 0.025, 0.975)
                Y_pseudo = D_test[:,-2]*(D_test[:,-1]/e_pred - (1-D_test[:,-1])/(1-e_pred))
                model2 = self.get_model()
                model2.fit(X=D_test[:,:-2],y=Y_pseudo)
                tau_k = lambda X: model2.predict(X=X)
                taus.append(tau_k)
            if self.median:
                tau = lambda X: np.median(np.array([tau_k(X) for tau_k in taus]), axis=0)
            else:
                tau = lambda X: np.mean(np.array([tau_k(X) for tau_k in taus]), axis=0)
            return tau

    def get_model(self, regressor=True):
        if regressor:
            model = RandomForestRegressor(
                    n_estimators=self.tree_settings['n_trees'],
                    max_features=self.tree_settings['max_features'],
                    min_samples_leaf=self.tree_settings['min_samples_leaf'],
                    max_depth=self.tree_settings['max_depth'],
                    n_jobs=-1)
        else:
            model = RandomForestClassifier(
                    n_estimators=self.tree_settings['n_trees'],
                    max_features=self.tree_settings['max_features'],
                    min_samples_leaf=self.tree_settings['min_samples_leaf'],
                    max_depth=self.tree_settings['max_depth'],
                    n_jobs=-1)
        return model

In [74]:
class DR_learner:
    def __init__(self, tree_settings=tree_settings_init,K=1,median=True):
        self.median = median
        self.tree_settings = tree_settings
        self.K = K

    def get_tau(self,D):
        if self.K == 1:
            model1 = self.get_model()
            model1.fit(X=D[D[:,-1] == 1,:-2],y=D[D[:,-1] == 1,-2])
            Y1_pred = model1.predict(D[:,:-2])
            model2 = self.get_model()
            model2.fit(X=D[D[:,-1] == 0,:-2],y=D[D[:,-1] == 0,-2])
            Y0_pred = model2.predict(D[:,:-2])
            model3 = self.get_model(regressor=False)
            model3.fit(X=D[:,:-2],y=D[:,-1])
            e_pred = np.clip(model3.predict_proba(D[:,:-2])[:,1], 0.025, 0.975)
            Y_pseudo = (D[:,-1]*(D[:,-2] - Y1_pred))/e_pred - ((1-D[:,-1])*(D[:,-2] - Y0_pred))/(1-e_pred) + Y1_pred - Y0_pred
            model4 = self.get_model()
            model4.fit(X=D[:,:-2],y=Y_pseudo)
            tau = lambda X: model4.predict(X=X)
            return tau
        else:
            kf = KFold(n_splits=self.K, shuffle=True)
            folds = [indices for _,indices in kf.split(D)]
            for k in range(self.K):
                taus = []
                D_test = D[folds[k]]
                D_train1 = D[folds[(k + 1) % self.K]]
                D_train2 = D[folds[(k + 2) % self.K]]
                model1 = self.get_model()
                model1.fit(X=D_train1[D_train1[:,-1] == 1,:-2],y=D_train1[D_train1[:,-1] == 1,-2])
                Y1_pred = model1.predict(D_test[:,:-2])
                model2 = self.get_model()
                model2.fit(X=D_train1[D_train1[:,-1] == 0,:-2],y=D_train1[D_train1[:,-1] == 0,-2])
                Y0_pred = model2.predict(D_test[:,:-2])
                model3 = self.get_model(regressor=False)
                model3.fit(X=D_train2[:,:-2],y=D_train2[:,-1])
                e_pred = np.clip(model3.predict_proba(D_test[:,:-2])[:,1], 0.025, 0.975)
                Y_pseudo = (D_test[:,-1]*(D_test[:,-2] - Y1_pred))/e_pred - ((1-D_test[:,-1])*(D_test[:,-2] - Y0_pred))/(1-e_pred) + Y1_pred - Y0_pred
                model4 = self.get_model()
                model4.fit(X=D_test[:,:-2],y=Y_pseudo)
                tau_k = lambda X: model4.predict(X=X)
                taus.append(tau_k)
            if self.median:
                tau = lambda X: np.median(np.array([tau_k(X) for tau_k in taus]), axis=0)
            else:
                tau = lambda X: np.mean(np.array([tau_k(X) for tau_k in taus]), axis=0)
            return tau

    def get_model(self, regressor=True):
        if regressor:
            model = RandomForestRegressor(
                    n_estimators=self.tree_settings['n_trees'],
                    max_features=self.tree_settings['max_features'],
                    min_samples_leaf=self.tree_settings['min_samples_leaf'],
                    max_depth=self.tree_settings['max_depth'],
                    n_jobs=-1)
        else:
            model = RandomForestClassifier(
                    n_estimators=self.tree_settings['n_trees'],
                    max_features=self.tree_settings['max_features'],
                    min_samples_leaf=self.tree_settings['min_samples_leaf'],
                    max_depth=self.tree_settings['max_depth'],
                    n_jobs=-1)
        return model

In [75]:
class X_learner:
    def __init__(self, tree_settings=tree_settings_init,K=1,median=True):
        self.median = median
        self.tree_settings = tree_settings
        self.K = K

    def get_tau(self,D):
        if self.K == 1:
            model1 = self.get_model()
            model1.fit(X=D[D[:,-1] == 1,:-2],y=D[D[:,-1] == 1,-2])
            Y1_pred = model1.predict(D[D[:,-1] == 0,:-2])

            model2 = self.get_model()
            model2.fit(X=D[D[:,-1] == 0,:-2],y=D[D[:,-1] == 0,-2])
            Y0_pred = model2.predict(D[D[:,-1] == 1,:-2])

            Y_pseudo1 = D[D[:,-1] == 1,-2] - Y0_pred
            model3 = self.get_model()
            model3.fit(X=D[D[:,-1] == 1,:-2],y=Y_pseudo1)

            Y_pseudo0 = Y1_pred - D[D[:,-1] == 0,-2]
            model4 = self.get_model()
            model4.fit(X=D[D[:,-1] == 0,:-2],y=Y_pseudo0)

            model5 = self.get_model(regressor=False)
            model5.fit(X=D[:,:-2],y=D[:,-1])
            tau = lambda X: np.clip(model5.predict_proba(X)[:,1], 0.025, 0.975)*model4.predict(X=X)+(1-np.clip(model5.predict_proba(X)[:,1], 0.025, 0.975))*model3.predict(X=X)
            return tau
        else:
            kf = KFold(n_splits=self.K, shuffle=True)
            folds = [indices for _,indices in kf.split(D)]
            for k in range(self.K):
                taus = []
                D_test = D[folds[k]]
                D_train1 = D[folds[(k + 1) % self.K]]
                D_train2 = D[folds[(k + 2) % self.K]]

                model1 = self.get_model()
                model1.fit(X=D_train1[D_train1[:,-1] == 1,:-2],y=D_train1[D_train1[:,-1] == 1,-2])
                Y1_pred = model1.predict(D_test[D_test[:,-1] == 0,:-2])

                model2 = self.get_model()
                model2.fit(X=D_train1[D_train1[:,-1] == 0,:-2],y=D_train1[D_train1[:,-1] == 0,-2])
                Y0_pred = model2.predict(D_test[D_test[:,-1] == 1,:-2])

                Y_pseudo1 = D_test[D_test[:,-1] == 1,-2] - Y0_pred
                model3 = self.get_model()
                model3.fit(X=D_test[D_test[:,-1] == 1,:-2],y=Y_pseudo1)

                Y_pseudo0 = Y1_pred - D_test[D_test[:,-1] == 0,-2]
                model4 = self.get_model()
                model4.fit(X=D_test[D_test[:,-1] == 0,:-2],y=Y_pseudo0)

                model5 = self.get_model(regressor=False)
                model5.fit(X=D_train2[:,:-2],y=D_train2[:,-1])

                tau_k = lambda X: np.clip(model5.predict_proba(X)[:,1], 0.025, 0.975)*model4.predict(X=X)+(1-np.clip(model5.predict_proba(X)[:,1], 0.025, 0.975))*model3.predict(X=X)
                taus.append(tau_k)
            if self.median:
                tau = lambda X: np.median(np.array([tau_k(X) for tau_k in taus]), axis=0)
            else:
                tau = lambda X: np.mean(np.array([tau_k(X) for tau_k in taus]), axis=0)
            return tau

    def get_model(self, regressor=True):
        if regressor:
            model = RandomForestRegressor(
                    n_estimators=self.tree_settings['n_trees'],
                    max_features=self.tree_settings['max_features'],
                    min_samples_leaf=self.tree_settings['min_samples_leaf'],
                    max_depth=self.tree_settings['max_depth'],
                    n_jobs=-1)
        else:
            model = RandomForestClassifier(
                    n_estimators=self.tree_settings['n_trees'],
                    max_features=self.tree_settings['max_features'],
                    min_samples_leaf=self.tree_settings['min_samples_leaf'],
                    max_depth=self.tree_settings['max_depth'],
                    n_jobs=-1)
        return model

In [76]:
class R_learner:
    def __init__(self, tree_settings=tree_settings_init,K=1,median=True):
        self.median = median
        self.tree_settings = tree_settings
        self.K = K

    def get_tau(self,D):
        if self.K == 1:
            model1 = self.get_model()
            model1.fit(X=D[:,:-2],y=D[:,-2])
            Y_pred = model1.predict(X=D[:,:-2])

            model2 = self.get_model(regressor=False)
            model2.fit(X=D[:,:-2],y=D[:,-1])
            e_pred = np.clip(model2.predict_proba(D[:,:-2])[:,1], 0.025, 0.975)

            Y_pseudo = (D[:,-2]-Y_pred)/(D[:,-1]-e_pred)
            model3 = self.get_model()
            model3.fit(X=D[:,:-2],y=Y_pseudo,sample_weight=(D[:,-1]-e_pred)**2)
            tau = lambda X: model3.predict(X=X)
            return tau
        else:
            kf = KFold(n_splits=self.K, shuffle=True)
            folds = [indices for _,indices in kf.split(D)]
            for k in range(self.K):
                taus = []
                D_test = D[folds[k]]
                D_train1 = D[folds[(k + 1) % self.K]]
                D_train2 = D[folds[(k + 2) % self.K]]
                model1 = self.get_model()
                model1.fit(X=D_train1[:,:-2],y=D_train1[:,-2])
                Y_pred = model1.predict(X=D_test[:,:-2])

                model2 = self.get_model(regressor=False)
                model2.fit(X=D_train2[:,:-2],y=D_train2[:,-1])
                e_pred = np.clip(model2.predict_proba(D_test[:,:-2])[:,1], 0.025, 0.975)

                Y_pseudo = (D_test[:,-2]-Y_pred)/(D_test[:,-1]-e_pred)
                model3 = self.get_model()
                model3.fit(X=D_test[:,:-2],y=Y_pseudo,sample_weight=(D_test[:,-1]-e_pred)**2)
                tau_k = lambda X: model3.predict(X=X)
                taus.append(tau_k)
            if self.median:
                tau = lambda X: np.median(np.array([tau_k(X) for tau_k in taus]), axis=0)
            else:
                tau = lambda X: np.mean(np.array([tau_k(X) for tau_k in taus]), axis=0)
            return tau

    def get_model(self, regressor=True):
        if regressor:
            model = RandomForestRegressor(
                    n_estimators=self.tree_settings['n_trees'],
                    max_features=self.tree_settings['max_features'],
                    min_samples_leaf=self.tree_settings['min_samples_leaf'],
                    max_depth=self.tree_settings['max_depth'],
                    n_jobs=-1)
        else:
            model = RandomForestClassifier(
                    n_estimators=self.tree_settings['n_trees'],
                    max_features=self.tree_settings['max_features'],
                    min_samples_leaf=self.tree_settings['min_samples_leaf'],
                    max_depth=self.tree_settings['max_depth'],
                    n_jobs=-1)
        return model

In [77]:
class Algorithm:
    def __init__(self, data=DGP().generate_data(), meta_learner='S_learner', K=1, median=True, tree_settings=tree_settings_init):
        """
        Algorithm class initializer.

        Parameters:
        data (np.ndarray): The generated data
        meta_learner (str): Name of the meta_learner ('S_learner' by default)
        K (int): Number of folds for cross-fitting (default is 5)
        option (int): Type of cross-fitting (1 or 2)
        tree_settings (dict): Dictionary of tree settings (default is tree_settings_init)
        """
        self.data = data
        self.K = K
        self.meta_learner = meta_learner
        self.median = median
        self.tree_settings = tree_settings

    def run(self):
        if self.meta_learner == 'S_learner':
            learner = S_learner(K=self.K, median=self.median, tree_settings=self.tree_settings)
        elif self.meta_learner == 'T_learner':
            learner = T_learner(K=self.K, median=self.median, tree_settings=self.tree_settings)
        elif self.meta_learner == 'IPW_learner':
            learner = IPW_learner(K=self.K, median=self.median, tree_settings=self.tree_settings)
        elif self.meta_learner == 'DR_learner':
            learner = DR_learner(K=self.K, median=self.median, tree_settings=self.tree_settings)
        elif self.meta_learner == 'X_learner':
            learner = X_learner(K=self.K, median=self.median, tree_settings=self.tree_settings)
        elif self.meta_learner == 'R_learner':
            learner = R_learner(K=self.K, median=self.median, tree_settings=self.tree_settings)
        else:
            raise ValueError(f'{self.meta_learner} is an invalid learner name')
        return learner.get_tau(self.data)

    def __repr__(self):
        """
        Return a string representation of the Algorithm object.
        """
        return f"{self.K}_{self.meta_learner}"

In [78]:
class Metrics:
    def mse(self, y_pred, y_true):
        """
        Calculate the Mean Squared Error (MSE).

        Parameters:
        y_pred (np.ndarray): Estimated CATE values (n x r)
        y_true (np.ndarray): True CATE values (n,)

        Returns:
        float: Mean Squared Error
        """
        # Calculate MSE for each observation
        mse_i = np.mean((y_pred - y_true[:, np.newaxis]) ** 2, axis=1)
        # Average over all observations
        mse = np.mean(mse_i)
        return mse

    def bias(self, y_pred, y_true):
        """
        Calculate the Absolute Bias.

        Parameters:
        y_pred (np.ndarray): Estimated CATE values (n x r)
        y_true (np.ndarray): True CATE values (n,)

        Returns:
        float: Absolute Bias
        """
        # Calculate mean prediction for each observation
        mean_y_pred = np.mean(y_pred, axis=1)
        # Calculate bias for each observation
        bias_i = np.abs(mean_y_pred - y_true)
        # Average over all observations
        bias = np.mean(bias_i)
        return bias

    def var(self, y_pred):
        """
        Calculate the Variance.

        Parameters:
        y_pred (np.ndarray): Estimated CATE values (n x r)

        Returns:
        float: Variance
        """
        # Calculate mean prediction for each observation
        mean_y_pred = np.mean(y_pred, axis=1)
        # Calculate variance for each observation
        var_i = np.mean((y_pred - mean_y_pred[:, np.newaxis]) ** 2, axis=1)
        # Average over all observations
        var = np.mean(var_i)
        return var

In [79]:
class Settings:
    def __init__(self, e=e_default, tau=tau_default, mu_0=mu_0_default, tree_settings=tree_settings_init, sigma_epsilon=1, alpha=0.5, n=1000, p=10, R=500):
        """
        Settings class initializer.

        Parameters:
        e (function): Propensity score function
        tau (function): Treatment effect function
        mu_0 (function): Baseline outcome function
        tree_settings (dict): Dictionary of tree settings
        alpha (float): Sparsity parameter for the covariance matrix
        n (int): Number of samples
        p (int): Number of features
        R (int): Number of repetitions
        K (int): Number of folds for cross-fitting
        meta_learner (str): Name of the meta_learner
        """
        # Random Forest tuning parameters
        self.tree_settings = tree_settings

        # Coefficients for mu_0 and e
        self.e = e
        self.mu_0 = mu_0
        self.tau = tau

        # Other settings
        self.sigma_epsilon = sigma_epsilon
        self.alpha = alpha
        self.n = n
        self.p = p
        self.R = R

    def update_settings(self, **kwargs):
        """
        Update settings dynamically.

        Parameters:
        kwargs: Key-value pairs of settings to update
        """
        for key, value in kwargs.items():
            if hasattr(self, key):
                setattr(self, key, value)
            else:
                raise KeyError(f"Invalid setting: {key}")

    def get_settings(self):
        """
        Retrieve current settings.

        Returns:
        dict: Dictionary of current settings
        """
        return {
            'tree_settings': self.tree_settings,
            'alpha': self.alpha,
            'sigma_epsilon': self.sigma_epsilon,
            'n': self.n,
            'p': self.p,
            'R': self.R
        }

    def __repr__(self):
        """
        String representation of the Settings object.

        Returns:
        str: Readable string representation of the current settings
        """
        settings = self.get_settings()
        return f"Settings({', '.join([f'{k}={v}' for k, v in settings.items()])})"

In [80]:
class Simulation:
    def __init__(self, name, settings, sample_sizes=2000, n_test=10000, R=100, meta_learners=None, options=None):
        self.name = name
        self.settings = settings
        self.sample_sizes = sample_sizes
        self.n_test = n_test
        self.R = R
        self.meta_learners = meta_learners if meta_learners else ['S_learner', 'T_learner', 'IPW_learner', 'DR_learner', 'X_learner', 'R_learner']
        self.options = options if options else [3]

    def run(self, save=True, folder='results', filename_prefix=None):
        # Generate test data
        dgp = DGP(n=self.n_test, p=self.settings.p, alpha=self.settings.alpha, sigma_epsilon=self.settings.sigma_epsilon, mu_0=self.settings.mu_0, tau=self.settings.tau, e=self.settings.e, seed=10000)
        test_data = dgp.generate_data()
        y_true = self.settings.tau(test_data)
        y_preds = {}
        metrics = {}

        for learner in self.meta_learners:
            for option in self.options:
                if option == 1:
                    y_preds[f'FS_{learner}'] = self.run_algorithm(learner, test_data, 1)
                elif option == 2:
                    y_preds[f'CF_{learner}'] = self.run_algorithm(learner, test_data, 5)
                else:
                    y_preds[f'CF_mean_{learner}'] = self.run_algorithm(learner, test_data, 5,median=False)


        # Calculate metrics
        metrics_calculator = Metrics()
        for alg, y_pred in y_preds.items():
            mse_value = metrics_calculator.mse(y_pred, y_true)
            bias_value = metrics_calculator.bias(y_pred, y_true)
            var_value = metrics_calculator.var(y_pred)
            metrics[alg] = {
                'MSE': mse_value,
                'Bias': bias_value,
                'Variance': var_value
            }
        # Save results
        if save:
            self.save_results(y_preds, y_true, metrics, filename_prefix, folder)
        return y_preds, y_true, metrics

    def run_algorithm(self, learner, test_data, K, median=True):
        y_preds = []
        print(f'{learner}_{K}')
        for r in range(self.settings.R):
            print(r)
            dgp_r = DGP(n=self.settings.n, p=self.settings.p, alpha=self.settings.alpha,sigma_epsilon=self.settings.sigma_epsilon, mu_0=self.settings.mu_0, tau=self.settings.tau, e=self.settings.e, seed=r)
            data = dgp_r.generate_data()
            algorithm = Algorithm(data=data, meta_learner=learner, K=K, median=median, tree_settings=self.settings.tree_settings)
            tau_hat = algorithm.run()
            y_pred = tau_hat(test_data[:, :-2])
            y_preds.append(y_pred)
        y_preds = np.transpose(np.array(y_preds))  # Transpose to get (n, R)
        return y_preds

    def save_results(self, y_preds, y_true, metrics, filename_prefix=None, folder='results'):
        # Create the folder if it doesn't exist
        print(folder)
        if not os.path.exists(folder):
            os.makedirs(folder)

        # Generate filename with date if prefix is not provided
        if filename_prefix is None:
            date_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            filename_prefix = f'{date_str}'

        # Save predictions
        y_preds_file = os.path.join(folder, f'{self.name}_{filename_prefix}_y_preds.pkl')
        with open(y_preds_file, 'wb') as f:
            pickle.dump(y_preds, f)
        print(f'Predictions saved to {y_preds_file}')

        # Save true values
        y_true_file = os.path.join(folder, f'{self.name}_{filename_prefix}_y_true.pkl')
        with open(y_true_file, 'wb') as f:
            pickle.dump(y_true, f)
        print(f'True values saved to {y_true_file}')

        # Save metrics
        metrics_file = os.path.join(folder, f'{self.name}_{filename_prefix}_metrics.pkl')
        with open(metrics_file, 'wb') as f:
            pickle.dump(metrics, f)
        print(f'Metrics saved to {metrics_file}')

In [81]:
# Default functions
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='baseline', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

S_learner_5
0
np.mean
1
np.mean
2
np.mean
3
np.mean
4
np.mean
5
np.mean
6
np.mean
7
np.mean
8
np.mean
9
np.mean
10
np.mean
11
np.mean
12
np.mean
13
np.mean
14
np.mean
15
np.mean
16
np.mean
17
np.mean
18
np.mean
19
np.mean
20
np.mean
21
np.mean
22
np.mean
23
np.mean
24
np.mean
25
np.mean
26
np.mean
27
np.mean
28
np.mean
29
np.mean
30
np.mean
31
np.mean
32
np.mean
33
np.mean
34
np.mean
35
np.mean
36
np.mean
37
np.mean
38
np.mean
39
np.mean
40
np.mean
41
np.mean
42
np.mean
43
np.mean
44
np.mean
45
np.mean
46
np.mean
47
np.mean
48
np.mean
49
np.mean
50
np.mean
51
np.mean
52
np.mean
53
np.mean
54
np.mean
55
np.mean
56
np.mean
57
np.mean
58
np.mean
59
np.mean
60
np.mean
61
np.mean
62
np.mean
63
np.mean
64
np.mean
65
np.mean
66
np.mean
67
np.mean
68
np.mean
69
np.mean
70
np.mean
71
np.mean
72
np.mean
73
np.mean
74
np.mean
75
np.mean
76
np.mean
77
np.mean
78
np.mean
79
np.mean
80
np.mean
81
np.mean
82
np.mean
83
np.mean
84
np.mean
85
np.mean
86
np.mean
87
np.mean
88
np.mean
89
np.mean
90
np.me

,CF_mean_S_learner,CF_mean_T_learner,CF_mean_IPW_learner,CF_mean_DR_learner,CF_mean_X_learner,CF_mean_R_learner
MSE,0.523719,0.132431,0.554809,0.224074,0.115717,0.208500
Bias,0.633157,0.239212,0.250992,0.166940,0.217062,0.174713
Variance,0.014113,0.035584,0.453537,0.175412,0.035670,0.153690


In [84]:
# Default functions
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=500,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='low_sample_size', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

S_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
S_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30

,FS_S_learner,CF_S_learner,FS_T_learner,CF_T_learner,FS_IPW_learner,CF_IPW_learner,FS_DR_learner,CF_DR_learner,FS_X_learner,CF_X_learner,FS_R_learner,CF_R_learner
MSE,0.515760,0.903514,0.141435,0.256685,0.438732,0.833110,0.157807,0.455901,0.134572,0.209363,0.191468,0.394023
Bias,0.623237,0.854865,0.247516,0.335678,0.422194,0.311796,0.202671,0.241440,0.231888,0.303447,0.172858,0.254871
Variance,0.015644,0.009998,0.039127,0.077841,0.172833,0.679044,0.086494,0.358553,0.044842,0.061318,0.138543,0.286225


In [85]:
# Default functions
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=5000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='high_sample_size', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

S_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
S_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30

,FS_S_learner,CF_S_learner,FS_T_learner,CF_T_learner,FS_IPW_learner,CF_IPW_learner,FS_DR_learner,CF_DR_learner,FS_X_learner,CF_X_learner,FS_R_learner,CF_R_learner
MSE,0.159529,0.337634,0.069147,0.110183,0.277079,0.393253,0.064600,0.147752,0.063112,0.096320,0.073434,0.145591
Bias,0.320276,0.490103,0.167290,0.224878,0.356526,0.214828,0.125194,0.140146,0.149194,0.201282,0.095690,0.147234
Variance,0.010902,0.011029,0.016801,0.023277,0.073001,0.315141,0.033280,0.112084,0.021479,0.026449,0.054307,0.105076


In [86]:
# Default functions
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=2000,
    p=5,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='low_covariate_space', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

S_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
S_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30

,FS_S_learner,CF_S_learner,FS_T_learner,CF_T_learner,FS_IPW_learner,CF_IPW_learner,FS_DR_learner,CF_DR_learner,FS_X_learner,CF_X_learner,FS_R_learner,CF_R_learner
MSE,0.101804,0.289883,0.077390,0.167454,0.416082,1.220879,0.119354,0.330292,0.074842,0.117681,0.172636,0.274397
Bias,0.193776,0.407182,0.104428,0.215989,0.321861,0.115962,0.064909,0.078475,0.086701,0.162288,0.051510,0.085961
Variance,0.031573,0.035551,0.047316,0.073515,0.247561,1.194012,0.107024,0.315090,0.053365,0.058891,0.166389,0.254304


In [87]:
# Default functions
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=2000,
    p=50,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='high_covariate_space', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

S_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
S_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30

,FS_S_learner,CF_S_learner,FS_T_learner,CF_T_learner,FS_IPW_learner,CF_IPW_learner,FS_DR_learner,CF_DR_learner,FS_X_learner,CF_X_learner,FS_R_learner,CF_R_learner
MSE,0.546262,0.910141,0.204526,0.298638,0.449084,0.535939,0.178703,0.303138,0.194940,0.260307,0.172506,0.300190
Bias,0.612867,0.813722,0.329297,0.407632,0.489846,0.402415,0.285275,0.326061,0.320187,0.376795,0.251802,0.334262
Variance,0.007256,0.007294,0.017628,0.028792,0.068961,0.279315,0.035364,0.128927,0.021559,0.027784,0.059894,0.114803


In [88]:
# Default functions
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return np.zeros(X.shape[0])

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='no_treatment_effect', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

S_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
S_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30

,FS_S_learner,CF_S_learner,FS_T_learner,CF_T_learner,FS_IPW_learner,CF_IPW_learner,FS_DR_learner,CF_DR_learner,FS_X_learner,CF_X_learner,FS_R_learner,CF_R_learner
MSE,0.000168,0.000191,0.015378,0.027615,0.084949,0.285164,0.037858,0.152756,0.020308,0.027758,0.069053,0.134745
Bias,0.006549,0.004848,0.042882,0.057714,0.062727,0.070697,0.037228,0.039050,0.026347,0.039409,0.045536,0.067244
Variance,0.000124,0.000167,0.013303,0.024170,0.076309,0.277105,0.035967,0.150412,0.019269,0.025837,0.066094,0.128851


In [14]:
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return np.sin(np.pi * X[:, 0] * X[:, 1]) + 2 * (X[:, 2] - 0.5)**2 + X[:, 3] + 0.5*X[:,4]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='complex_treatment_effect', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

S_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
S_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30

,FS_S_learner,CF_S_learner,FS_T_learner,CF_T_learner,FS_IPW_learner,CF_IPW_learner,FS_DR_learner,CF_DR_learner,FS_X_learner,CF_X_learner,FS_R_learner,CF_R_learner
MSE,7.948124,12.144695,5.809914,9.150525,5.335038,5.980632,4.212993,5.299093,5.208384,7.896220,3.018299,5.301718
Bias,1.752102,2.182774,1.565032,2.039652,1.441306,1.502634,1.292346,1.424430,1.473427,1.873575,1.083293,1.429384
Variance,0.114409,0.061641,0.235767,0.194589,0.350102,1.222057,0.283873,0.595177,0.228852,0.192521,0.381415,0.561599


In [15]:
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': 10,
    'min_samples_leaf': 10,
    'max_features': 'log2'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='shallow_trees', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

S_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
S_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30

,FS_S_learner,CF_S_learner,FS_T_learner,CF_T_learner,FS_IPW_learner,CF_IPW_learner,FS_DR_learner,CF_DR_learner,FS_X_learner,CF_X_learner,FS_R_learner,CF_R_learner
MSE,0.290576,0.595451,0.101752,0.158764,0.230693,0.423049,0.091571,0.200576,0.091227,0.137287,0.095153,0.190026
Bias,0.451687,0.673154,0.216095,0.283500,0.294214,0.273968,0.155663,0.196450,0.193923,0.258345,0.148314,0.207065
Variance,0.009188,0.010562,0.018789,0.026559,0.086649,0.302339,0.044882,0.134100,0.024548,0.026133,0.054163,0.116051


In [ ]:
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 2,
    'max_features': None
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='deep_trees', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

In [ ]:
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=1.15, c2=0.1):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='low_confounding', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

In [ ]:
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=7.525, c2=0.1):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='high_confounding', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

In [ ]:
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.92,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='low_correlation', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

In [ ]:
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.48,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='high_correlation', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

In [ ]:
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.35, c2=np.log(1)):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='balanced', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

In [ ]:
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.66, c2=np.log(19)):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='imbalanced', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

In [65]:
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=0.45,
    alpha=0.85,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='low_noise', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

S_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
S_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30

,FS_S_learner,CF_S_learner,FS_T_learner,CF_T_learner,FS_IPW_learner,CF_IPW_learner,FS_DR_learner,CF_DR_learner,FS_X_learner,CF_X_learner,FS_R_learner,CF_R_learner
MSE,0.207252,0.463790,0.072263,0.116515,0.303922,0.467263,0.060198,0.129533,0.062759,0.098176,0.057264,0.121027
Bias,0.380559,0.594688,0.177637,0.239845,0.377400,0.250407,0.136048,0.157026,0.159746,0.216394,0.106059,0.165574
Variance,0.007986,0.008776,0.013759,0.019371,0.080853,0.366059,0.022286,0.085551,0.014713,0.018272,0.032556,0.070161


In [66]:
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + 0.5 * X[:, 0] + 0.3 * X[:,1]

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=2.07,
    alpha=0.85,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='high_noise', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

S_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
S_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30

,FS_S_learner,CF_S_learner,FS_T_learner,CF_T_learner,FS_IPW_learner,CF_IPW_learner,FS_DR_learner,CF_DR_learner,FS_X_learner,CF_X_learner,FS_R_learner,CF_R_learner
MSE,0.307017,0.682731,0.120763,0.203587,0.428384,0.910325,0.195526,0.601964,0.134625,0.189022,0.297277,0.558653
Bias,0.460983,0.729130,0.173159,0.244687,0.379450,0.261201,0.142247,0.190726,0.160089,0.224783,0.125408,0.195169
Variance,0.027807,0.023759,0.065192,0.103163,0.202059,0.801559,0.156996,0.540830,0.087842,0.104604,0.268098,0.493252


In [16]:
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return np.ones(X.shape[0])

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='independent', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

S_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
S_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30

,FS_S_learner,CF_S_learner,FS_T_learner,CF_T_learner,FS_IPW_learner,CF_IPW_learner,FS_DR_learner,CF_DR_learner,FS_X_learner,CF_X_learner,FS_R_learner,CF_R_learner
MSE,0.106509,0.309893,0.015379,0.027605,0.208299,0.392822,0.037841,0.152818,0.020307,0.027750,0.067030,0.135577
Bias,0.314550,0.545920,0.042870,0.057726,0.303342,0.086603,0.037196,0.039101,0.026344,0.039411,0.034292,0.066228
Variance,0.007187,0.011775,0.013304,0.024158,0.097376,0.379879,0.035955,0.150467,0.019269,0.025829,0.065262,0.129860


In [17]:
def mu_0(X, c1=2.25, c2=1.0):
    """Baseline outcome function."""
    return np.exp(c1 * X[:, 0]) / (1 + np.exp(c1 * X[:, 0])) + c2 * X[:, 2]

def e(X, c1=0.3775, c2=0.01):
    """Propensity score function."""
    a = c1 * X[:, 0] + c2 * np.exp(X[:, 2]) / (1 + np.exp(X[:, 2]))
    return 1 / (1 + np.exp(a))

def tau(X):
    """Treatment effect function."""
    return 1 + np.sum([(1 / (j+2)) * X[:, j] for j in range(15)], axis=0)

# Assuming the necessary imports and function definitions (e_default, tau_default, mu_0_default, tree_settings_init) are already in place
tree_settings = {
    'n_trees': 500,
    'max_depth': None,
    'min_samples_leaf': 5,
    'max_features': 'sqrt'
}

default_settings = Settings(
    e=e,
    tau=tau,
    mu_0=mu_0,
    tree_settings=tree_settings,
    sigma_epsilon=1,
    alpha=0.85,
    n=2000,
    p=20,
    R=100
)

# Initialize the simulation with default settings
simulation = Simulation(name='highly_dependent', settings=default_settings)

# Record the start time
start_time = time.time()

# Run the simulation
_, _, metrics = simulation.run()

# Record the end time
end_time = time.time()

# Calculate the elapsed time
elapsed_time = end_time - start_time

print(f"Simulation completed in {elapsed_time:.2f} seconds")
pd.DataFrame(metrics)

S_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
S_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
T_learner_5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30

,FS_S_learner,CF_S_learner,FS_T_learner,CF_T_learner,FS_IPW_learner,CF_IPW_learner,FS_DR_learner,CF_DR_learner,FS_X_learner,CF_X_learner,FS_R_learner,CF_R_learner
MSE,0.370084,0.673143,0.206351,0.272136,0.408134,0.630035,0.186641,0.323370,0.186876,0.239915,0.189220,0.308668
Bias,0.484494,0.682183,0.333691,0.387295,0.432036,0.333085,0.282202,0.287623,0.308224,0.357373,0.242712,0.296865
Variance,0.014336,0.012609,0.028832,0.035020,0.118020,0.456178,0.058773,0.192305,0.035099,0.037706,0.094226,0.167856
